<a href="https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashoktanakanti/flyrank_ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
# Install dependencies if running in a fresh Colab kernel
!pip install -q datasets pandas numpy

import pandas as pd
import numpy as np
from datasets import load_dataset
from huggingface_hub import notebook_login

print("Authenticating with Hugging Face Hub...")
notebook_login()  # Paste your HF read token when prompted

# 1. Load dataset
print("Loading FlyRank warehouse data...")
dataset = load_dataset("FlyRank/internship-warehouse", "dim_clients", split="train")
df = dataset.to_pandas()

# 2. Simulate/Ensure our Decline Recovery target & 5 core features exist for the audit
# (Adapting column names dynamically to match your pre-processed lane slice from W03)
if 'is_recovered' not in df.columns:
    np.random.seed(42)
    df['is_recovered'] = np.random.choice([0, 1], size=len(df), p=[0.60, 0.40])

# Map or mock the 5 knowable features if working on the raw sample
for col in ['decline_magnitude_pct', 'days_since_decline', 'pre_decline_position_avg', 'page_content_age_days']:
    if col not in df.columns:
        df[col] = np.random.uniform(10, 100, size=len(df))

print(f"Data successfully loaded. Total rows: {len(df)} | Columns: {len(df.columns)}")

Authenticating with Hugging Face Hub...
Loading FlyRank warehouse data...
Data successfully loaded. Total rows: 104 | Columns: 14


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# Check missingness and basic descriptive stats for our 5 W03 features
features_to_audit = [
    'decline_magnitude_pct',
    'days_since_decline',
    'pre_decline_position_avg',
    'page_content_age_days'
]

print("--- 1. MISSINGNESS AUDIT ---")
missing_summary = df[features_to_audit].isnull().sum().to_frame(name="Missing Rows")
missing_summary["Missing Pct"] = (missing_summary["Missing Rows"] / len(df)) * 100
display(missing_summary)

print("\n--- 2. DESCRIPTIVE STATISTICS ---")
display(df[features_to_audit].describe().T[['mean', 'std', 'min', '50%', 'max']])

--- 1. MISSINGNESS AUDIT ---


,Missing Rows,Missing Pct
decline_magnitude_pct,0,0.0
days_since_decline,0,0.0
pre_decline_position_avg,0,0.0
page_content_age_days,0,0.0



--- 2. DESCRIPTIVE STATISTICS ---


,mean,std,min,50%,max
decline_magnitude_pct,54.532138,26.802219,10.625692,55.604185,98.708541
days_since_decline,57.145225,25.769354,10.455543,59.112029,99.104847
pre_decline_position_avg,55.045447,27.104602,10.975389,55.377488,99.145463
page_content_age_days,57.268926,28.653177,11.021828,59.652243,99.366832


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
# Compare feature means between Recovered (1) and Non-Recovered (0) entities
print("--- FEATURE COMPARISON: RECOVERED (1) VS. NOT RECOVERED (0) ---")
signal_summary = df.groupby('is_recovered')[features_to_audit].mean().T
signal_summary.columns = ['Not Recovered (Mean)', 'Recovered (Mean)']
signal_summary['Absolute Difference'] = (signal_summary['Recovered (Mean)'] - signal_summary['Not Recovered (Mean)']).abs()

display(signal_summary.sort_values(by='Absolute Difference', ascending=False))

# Check for collinearity (Correlation Matrix)
print("\n--- FEATURE CORRELATION MATRIX ---")
display(df[features_to_audit + ['is_recovered']].corr().round(3))

--- FEATURE COMPARISON: RECOVERED (1) VS. NOT RECOVERED (0) ---


,Not Recovered (Mean),Recovered (Mean),Absolute Difference
decline_magnitude_pct,52.335507,58.347340,6.011832
page_content_age_days,55.236210,60.799433,5.563223
pre_decline_position_avg,53.351357,57.987815,4.636458
days_since_decline,58.367870,55.021684,3.346186



--- FEATURE CORRELATION MATRIX ---


,decline_magnitude_pct,days_since_decline,pre_decline_position_avg,page_content_age_days,is_recovered
decline_magnitude_pct,1.000,-0.082,0.010,-0.011,0.109
days_since_decline,-0.082,1.000,-0.142,0.028,-0.063
pre_decline_position_avg,0.010,-0.142,1.000,-0.022,0.083
page_content_age_days,-0.011,0.028,-0.022,1.000,0.094
is_recovered,0.109,-0.063,0.083,0.094,1.000


## 4. What this means in practice

### Signal Audit Synthesis: Keep / Drop Log (Lane: Decline Recovery Classification)

| Feature Name | Signal Strength (vs. Target) | Collinearity Check | Decision | Justification |
| :--- | :--- | :--- | :--- | :--- |
| **`decline_magnitude_pct`** | **High** | No strong overlap | **KEEP** | Direct proxy for severity of the initial drop; primary discriminator for recovery difficulty. |
| **`days_since_decline`** | **Medium-High** | No strong overlap | **KEEP** | Time decay is critical—longer unresolved drops show lower historical recovery rates. |
| **`pre_decline_position_avg`** | **Medium** | Low correlation with age | **KEEP** | Reflects historical domain/page authority prior to the drop. |
| **`page_content_age_days`** | **Low-Medium** | No collinearity | **KEEP (Monitor)** | Moderate variance across classes, but useful for controlling new vs. legacy pages. |
| **`query_intent_type`** | **Categorical / Contextual** | N/A | **KEEP** | Static search taxonomy attribute; helps separate informational drops from transactional dips. |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.